In [32]:
import pandas as pd

df_sortimento = pd.read_excel('data/SORTIMENTO PARA FCT ALONGADO.xlsx', engine='calamine')
df_sortimento.head()

,LOJA,PRODUTO,COR,TAMANHO,MINIMO,ALVO,DIAS_ESTOQUE,VENDAS,DIAS_VENDA,VELOCIDADE_VENDA,COBERTURA,VEL_VDA_COB,RESSUPRIMENTO
0,4,0241,N0A00S,XP,0,NaN,NaN,NaN,NaN,NaN,122,NaN,NÃO
1,4,0241,N0A00S,P,3,NaN,NaN,NaN,NaN,NaN,122,NaN,NÃO
2,4,0241,N0A00S,M,4,NaN,NaN,NaN,NaN,NaN,122,NaN,NÃO
3,4,0241,N0A00S,G,4,NaN,NaN,NaN,NaN,NaN,122,NaN,NÃO
4,4,0241,N0A00S,XG,2,NaN,NaN,NaN,NaN,NaN,122,NaN,NÃO


In [35]:
df_sortimento['ARTIGO_COR'] = df_sortimento['PRODUTO']  + df_sortimento['COR']

In [36]:
lista_artigos_unicos = df_sortimento['ARTIGO_COR'].dropna().unique() 
string_artigos = ','.join(lista_artigos_unicos)


lista_filiais_unicas = df_sortimento['LOJA'].dropna().astype(str).unique()
string_filiais = ','.join(lista_filiais_unicas)

In [37]:
print("Tamanho da lista de Artigos: ", len(lista_artigos_unicos))
print("Tamanho da lista de Filiais: ", len(lista_filiais_unicas))

Tamanho da lista de Artigos:  160
Tamanho da lista de Filiais:  632


In [38]:
from google.cloud import bigquery
import warnings
warnings.filterwarnings("ignore", message="Your application has authenticated using end user credentials")



client = bigquery.Client(project='hering-data-analytics-prod')


query = f"""


DECLARE lista_artigos STRING;
DECLARE lista_filial  STRING;

SET lista_artigos = '{string_artigos}';
SET lista_filial = '{string_filiais}';



with cte as (
SELECT 
  cast(c.PontoVda as STRING) as PontoVda,
  c.Sit,
  c.Usuario,
  c.ARTIGO_COR,
  c.Tamanho,
  sum(cast(c.Pecas as integer)) as Pecas
FROM `hering-data-analytics-prod.BI.vendas_sell_in_fato_bi` c
WHERE  c.Sit In ('PENDENTE', 'ATENDIDO', 'SUSPENSO COMERCIAL', 'SUSPENSO COMERCIAL + FINANCEIRO', 'SUSPENSO FINANCEIRO') and  DATE_TRUNC(DATE(DataHora_Criacao_OV, 'America/Sao_Paulo'), MONTH) = DATE_TRUNC(CURRENT_DATE('America/Sao_Paulo'), MONTH) and c.Usuario = 'ABF'
and c.ARTIGO_COR  IN UNNEST(SPLIT(lista_artigos, ','))
and CAST(c.PontoVda AS STRING) IN UNNEST(SPLIT(lista_filial, ','))
group by c.PontoVda, c.Sit, c.Usuario, c.ARTIGO_COR, c.Tamanho
order by c.PontoVda asc
)

select * from cte where PontoVda IS NOT NULL 

"""

# 4. Executa e joga direto para um DataFrame do Pandas
print("Rodando a query...")
df_CA = client.query(query).to_dataframe()

# 5. Visualiza o resultado
print(f"Total de linhas retornadas: {len(df_CA)}")
display(df_CA.head())

Rodando a query...
Total de linhas retornadas: 56024


,PontoVda,Sit,Usuario,ARTIGO_COR,Tamanho,Pecas
0,1,ATENDIDO,ABF,K5AGNMCSI,M,1
1,1,ATENDIDO,ABF,KZHA1DSN,M,3
2,1,ATENDIDO,ABF,KG2AAX7SI,XG,1
3,1,PENDENTE,ABF,KTNUNNJSI,040,1
4,1,PENDENTE,ABF,761311BEN,G,1


In [39]:
df_CA['PontoVda'].dtypes

dtype('O')

In [6]:
df_CA['PontoVda'].nunique()


499

In [7]:
df_CA['ARTIGO_COR'].nunique()

160

In [8]:
df_CA.to_excel('data/df_CA.xlsx', index=False)

In [20]:
query = f"""

DECLARE lista_artigos STRING;
DECLARE lista_filial  STRING;

SET lista_artigos = '{string_artigos}';
SET lista_filial = '{string_filiais}';




SELECT
    A.FilialID,
    A.ArtigoID ARTIGO_COR,
    A.Tamanho,
    A.Data,
    B.TIPO_FILIAL as CANAL,
    D.PONTO_VENDA_COD AS FILIAL,
    cast(coalesce(SUM(A.Q_Pecas), 0) as INT) AS total_pecas,
    coalesce(SUM(A.Q_Estoque_Total), 0) AS estoque_total
FROM
    `hering-data-analytics-prod.BI.performance_comercial_fato_bi` AS A
LEFT JOIN
    `hering-data-analytics-prod.GOLD.dim_filiais_ro` AS B
    ON A.FilialID = B.LOJA_COD
LEFT JOIN 
    `hering-data-analytics-prod.BI.performance_comercial_dim_filiais` as D on A.FilialID = D.LOJA_ID
WHERE
 A.Data = CURRENT_DATE() - 1
    AND B.TIPO_FILIAL = 'FRANQUIAS' 
    AND A.ArtigoID IN UNNEST(SPLIT(lista_artigos, ','))
    AND D.PONTO_VENDA_COD IN UNNEST(SPLIT(lista_filial, ','))
GROUP BY
    A.FilialID,
    A.ArtigoID,
    A.Tamanho,
    A.Data,
    A.Piramide,
    B.TIPO_FILIAL,
    B.PONTO_VENDA,
    D.PONTO_VENDA_COD
ORDER BY
    A.FilialID ASC


"""

# 4. Executa e joga direto para um DataFrame do Pandas
print("Rodando a query...")
df_stk  = client.query(query).to_dataframe()

# 5. Visualiza o resultado
print(f"Total de linhas retornadas: {len(df_stk)}")
display(df_stk.head())

Rodando a query...
Total de linhas retornadas: 375126


,FilialID,ARTIGO_COR,Tamanho,Data,CANAL,FILIAL,total_pecas,estoque_total
0,20,KZF47ASN,048,2026-01-29,FRANQUIAS,104,0,1
1,20,022BN1007S,XXG,2026-01-29,FRANQUIAS,104,0,1
2,20,036H7QEN,G,2026-01-29,FRANQUIAS,104,0,3
3,20,KTNUNNJSI,050,2026-01-29,FRANQUIAS,104,0,1
4,20,022BN1007S,EXG,2026-01-29,FRANQUIAS,104,0,1


In [21]:
df_stk.shape

(375126, 8)

In [40]:
df_stk['FILIAL'].dtypes

dtype('O')

In [22]:
df_stk['estoque_total'].sum()


np.int64(1480577)

In [23]:
df_stk['Estoque Total'] = (df_stk['estoque_total'] - df_stk['total_pecas']).abs()

In [24]:
df_stk.head()

,FilialID,ARTIGO_COR,Tamanho,Data,CANAL,FILIAL,total_pecas,estoque_total,Estoque Total
0,20,KZF47ASN,048,2026-01-29,FRANQUIAS,104,0,1,1
1,20,022BN1007S,XXG,2026-01-29,FRANQUIAS,104,0,1,1
2,20,036H7QEN,G,2026-01-29,FRANQUIAS,104,0,3,3
3,20,KTNUNNJSI,050,2026-01-29,FRANQUIAS,104,0,1,1
4,20,022BN1007S,EXG,2026-01-29,FRANQUIAS,104,0,1,1


In [25]:
df_agregado = df_stk.groupby('Data')[['Estoque Total']].sum().reset_index()
display(df_agregado)

,Data,Estoque Total
0,2026-01-29,1503097


In [26]:
df_agregado = df_stk.groupby('Data')[['total_pecas']].sum().reset_index()
display(df_agregado)

,Data,total_pecas
0,2026-01-29,10320


In [27]:
df_stk.isnull().sum()

FilialID         0
ARTIGO_COR       0
Tamanho          1
Data             0
CANAL            0
FILIAL           0
total_pecas      0
estoque_total    0
Estoque Total    0
dtype: int64

In [28]:
df_stk['Tamanho'].unique()

array(['048', 'XXG', 'G', '050', 'EXG', 'M', 'XG', '046', '042', '044',
       '038', 'P', '036', '040', 'M       ', 'XG ', '034', 'XP', '052',
       'XG      ', 'XXG     ', 'P  ', 'M  ', 'G  ', '006', '008', '004',
       '012', '016', '010', 'P       ', '014', '001', '002', 'G       ',
       '005', 'XP ', '003', None, 'G1', '054', '056', 'G3'], dtype=object)

In [29]:
df_stk['ARTIGO_COR'].nunique()

160

In [30]:
df_stk.drop(columns=['estoque_total', 'total_pecas'], inplace= True)
df_stk.head()

,FilialID,ARTIGO_COR,Tamanho,Data,CANAL,FILIAL,Estoque Total
0,20,KZF47ASN,048,2026-01-29,FRANQUIAS,104,1
1,20,022BN1007S,XXG,2026-01-29,FRANQUIAS,104,1
2,20,036H7QEN,G,2026-01-29,FRANQUIAS,104,3
3,20,KTNUNNJSI,050,2026-01-29,FRANQUIAS,104,1
4,20,022BN1007S,EXG,2026-01-29,FRANQUIAS,104,1


In [31]:
df_stk.to_excel('data/df_STK.xlsx', index=False)